# Unit 00 · Setup and the workflow (≈30 min)

Goal: the same notebook runs **here, on Colab and on Kaggle**, and you've practised the loop once on a toy:
**stub → your code → check → move into `src/xray3d` → pytest → log**.

Watch first: **video 00, "The big picture"**. Run cells top to bottom with `Shift+Enter`.

In [ ]:
# --- setup (written by Claude, no need to change it): the same cell works locally, on Colab and on Kaggle ---
import os, sys, subprocess
from pathlib import Path

UNIT = "00-setup"
REPO_URL = "https://github.com/AdirBuskila/pelvis-xray-3d"
IN_CLOUD = "google.colab" in sys.modules or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_CLOUD:
    root = Path("/content" if "google.colab" in sys.modules else "/kaggle/working") / "pelvis-xray-3d"
    if not root.exists():  # Kaggle: turn Internet on in the notebook settings first
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:  # local: the repo is the first folder above this notebook that has a pyproject.toml
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root / "units" / UNIT)
for p in (root / "src", Path.cwd()):  # import xray3d (even before `pip install -e .`) and this unit's checks
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import xray3d
from xray3d import data, env, plotting
import test_unit as T

env.summary()

## 1 · What you just saw

* **running on** `local` here. On Colab it says `colab`, on Kaggle `kaggle`.
* **GPU** is `no` on the laptop (AMD integrated graphics: PyTorch can't use it). Training goes to Colab/Kaggle.
* **data root** is where datasets live. It's in `.gitignore`: data never goes to GitHub (size and licences).

## 2 · The same notebook in the cloud

Do this once, together, and write what you saw in `LOG.md`.

1. **Colab:** open
   [colab.research.google.com/github/AdirBuskila/pelvis-xray-3d/blob/main/units/00-setup/notebook.ipynb](https://colab.research.google.com/github/AdirBuskila/pelvis-xray-3d/blob/main/units/00-setup/notebook.ipynb)
   and run the setup cell. *Runtime → Change runtime type → T4 GPU* gives a GPU; run it again and the summary says `GPU: yes`.
2. **Kaggle:** *Create → New notebook → File → Import notebook → GitHub*, paste the repo URL, pick this notebook.
   In the right panel turn **Internet on** (the setup cell clones the repo), and pick *Accelerator → GPU T4 x2* when you need one.

Both clone the **public** repo, so they see only what was pushed. Changes you make there are lost unless you
download the notebook; the real work happens here, locally.

## 3 · Warm-up: shadows

An X-ray is a shadow: every ray adds up what it passes through. Here's the same idea in 2D, on a tiny image
of a square. A "ray" is a column (axis 0) or a row (axis 1), and pixels are `pixel_mm` wide.

In [ ]:
img = T.square_image()          # 10 x 10 zeros with a 4 x 4 square of ones
print(img)
plotting.show(img, "a 4 x 4 square", vmin=0, vmax=1)

### TODO 0.1 · `shadow`

Write `shadow(img, axis, pixel_mm)`: for every ray, **how many millimetres of object** does it cross?
`axis=0` means the rays run down the columns (one value per column).

> **Nudge:** a ray along axis 0 sees one column. What do you do to a column of 0s and 1s to count the 1s, and what turns a count of pixels into millimetres?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def shadow(img, axis, pixel_mm):
    """Path length (mm) through the object along each ray.

    img      : (H, W) array of 0/1, 1 = object
    axis     : 0 -> rays run down the columns (result has W values); 1 -> along the rows (H values)
    pixel_mm : pixel size in mm
    returns  : 1D float array
    """
    raise NotImplementedError("TODO 0.1")

In [ ]:
T.check_shadow(shadow)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
plotting.show(img, "image", ax=ax[0], vmin=0, vmax=1)
ax[1].bar(range(img.shape[1]), shadow(img, 0, 1.0)); ax[1].set_title("shadow down the columns (axis 0)"); ax[1].set_ylabel("mm")
ax[2].bar(range(img.shape[0]), shadow(img, 1, 1.0)); ax[2].set_title("shadow along the rows (axis 1)")
plt.tight_layout(); plt.show()

### TODO 0.2 · `same_shadow`

Video 00 says one X-ray loses depth: **different shapes can cast the same shadow**. Prove it.
Return a *different* 0/1 image that casts exactly the same shadow as the square along axis 0.

> **Nudge:** the shadow along axis 0 only counts how many 1s each column has. Does it care where in the column they are?

_Stuck? Ask Claude for **hint 2** (the approach) or **hint 3** (partial code). Every hint used goes in `LOG.md`._

In [ ]:
def same_shadow(img):
    """A different 0/1 image with the same column sums (same shadow along axis 0)."""
    raise NotImplementedError("TODO 0.2")

In [ ]:
T.check_same_shadow(same_shadow)

In [ ]:
other = same_shadow(img)
plotting.show_row([img, other], ["the square", "your shape"], vmin=0, vmax=1, size=3.5)
print("axis 0 shadows:", shadow(img, 0, 1.0), shadow(other, 0, 1.0), sep="\n  ")
print("axis 1 shadows:", shadow(img, 1, 1.0), shadow(other, 1, 1.0), sep="\n  ")

**Explain-back (say it out loud, then write one line each in `LOG.md`):**

1. Your shape fools the first view. Does it fool the second view (axis 1)? What does that say about taking two X-rays?
2. Can you build a shape that fools **both** views? (Try on paper, a 3×3 grid is enough.) If yes, what else
   does a method need, besides two views, to pick the right answer? This is why the project uses a network
   that has seen hundreds of pelvises.

## 4 · Move your code into the package and run pytest

1. Open `src/xray3d/toy.py`, paste `shadow` and `same_shadow` (with their docstrings).
2. Run the cell below. Both tests should **pass** (before the move they were **skipped**).

In [ ]:
!{sys.executable} -m pytest -q .

## 5 · Close the session

Fill in `LOG.md` (who drove which TODO, hints used, explain-back answers). Then ask Claude to commit and push.
Next: **unit 01**, after video 01.